In [ ]:
!pip install torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements.txt

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
import torch
torch.backends.cudnn.benchmark = True 
import sys
from transformers import DataCollatorWithPadding, AutoTokenizer
from torch.utils.data import DataLoader
import gc
from pytorch_lightning.loggers import TensorBoardLogger
from datasets import Dataset as HFDataset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
sys.modules.pop("models", None)
sys.modules.pop("data_preparation", None)
from functionality.data_preparation import  CustomDataset
from functionality.models import MolFormerClassifier, ClearMemoryCallback

In [6]:
MAX_LENGTH = 150

def load_data(train_data_path, val_data_path):
    train_data = pd.read_parquet(train_data_path)[['molecule_smiles', 'binds']]
    val_data = pd.read_parquet(val_data_path)[['molecule_smiles', 'binds']]
    return train_data, val_data

# Tokenization function
def tokenize_function(samples, tokenizer):
    tokenized = tokenizer(
        samples['molecule_smiles'],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )
    tokenized['labels'] = samples['binds']
    return tokenized

# Prepare DataLoaders
def prepare_dataloaders(protein_name, train_data, val_data, tokenizer, BATCH_SIZE = 1000):
    train_dataset = HFDataset.from_pandas(train_data)
    val_dataset = HFDataset.from_pandas(val_data)


    train_tokenized_path = f'{protein_name}_train_tokenized.parquet'
    val_tokenized_path = f'{protein_name}_val_tokenized.parquet'
    if os.path.exists(train_tokenized_path):
        print(f"Loading existing tokenized training data for {protein_name}")
        train_tokenized = pd.read_parquet(train_tokenized_path)
        train_tokenized = train_tokenized.reset_index(drop=True)  
    else:
        print(f"Tokenizing and saving training data for {protein_name}")
        train_tokenized = train_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": tokenizer})
        train_df = pd.DataFrame(train_tokenized)
        train_df.to_parquet(train_tokenized_path)
    
    if os.path.exists(val_tokenized_path):
        print(f"Loading existing tokenized validation data for {protein_name}...")
        val_tokenized = pd.read_parquet(val_tokenized_path)
        val_tokenized = val_tokenized.reset_index(drop=True) 
    else:
        print(f"Tokenizing and saving validation data for {protein_name}...")
        val_tokenized = val_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": tokenizer})
        val_df = pd.DataFrame(val_tokenized)
        val_df.to_parquet(val_tokenized_path)


    train_custom_dataset = CustomDataset(train_tokenized)
    val_custom_dataset = CustomDataset(val_tokenized)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    num_workers = torch.cuda.device_count() * 8
    train_dataloader = DataLoader(
        train_custom_dataset,
        shuffle=True,
        batch_size=BATCH_SIZE,
        num_workers=num_workers,
        collate_fn=data_collator
    )
    val_dataloader = DataLoader(
        val_custom_dataset,
        batch_size=BATCH_SIZE,
        num_workers=num_workers,
        collate_fn=data_collator
    )

    return train_dataloader, val_dataloader

In [8]:
def training_molformer_model(protein_name, tokenizer):

    train_data_path = f'train_data/{protein_name}/{protein_name}_train.parquet'
    val_data_path = f'train_data/{protein_name}/{protein_name}_val.parquet'
    train_data, val_data = load_data(train_data_path, val_data_path)
    train_dataloader, val_dataloader = prepare_dataloaders(protein_name, train_data, val_data, tokenizer)
    
    model_name = 'MolFormer'
    model = MolFormerClassifier(learning_rate=2e-5)
    
    torch.set_float32_matmul_precision("medium")
    logger = TensorBoardLogger("logs", name="molformer_binary_classification")
    clear_memory_callback = ClearMemoryCallback()
    early_stopping = EarlyStopping(monitor="val_loss", patience=2, mode="min")
    checkpoint_callback = ModelCheckpoint(
        dirpath='/models/',
        filename=f"{protein_name}_{model_name}",
        monitor="val_loss",
        save_top_k=1,
        mode="min",
        verbose=True,
    )

    
    trainer = pl.Trainer(
        max_epochs=5,
        accelerator="auto",
        devices=1,
        log_every_n_steps=2,
        callbacks=[early_stopping, checkpoint_callback, clear_memory_callback],
        logger=logger,
        gradient_clip_val=1.0
    )
    
    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)

    gc.collect()
    torch.cuda.empty_cache()
        

In [ ]:
protein_names = ['sEH', 'BRD4', 'HSA']
tokenizer = AutoTokenizer.from_pretrained('ibm/MoLFormer-XL-both-10pct', trust_remote_code=True)

for protein in protein_names:
    training_molformer_model(protein, tokenizer)